# TensorSeries

A `TensorSeries` is the `pd.Series` subclass that `NestedFrame` hands back for a tensor column, the way it hands back a `NestedSeries` for a nested column. Every row holds one numpy array of the same shape (an image cutout, a spectrum, an embedding), stored as a pyarrow `fixed_shape_tensor`. This notebook walks through what it adds on top of a plain Series.

The sections up to "Setting tensors" are on the `sean/tensor_series` branch; the rest is on `sean/tensor_series_ops`.

In [1]:
import numpy as np
import pandas as pd

from nested_pandas import NestedFrame, NestedSeries, TensorSeries
from nested_pandas.tensors import TensorExtensionArray

## Getting a TensorSeries from a NDF

Build a `NestedFrame` with two tensor columns and a nested column. The images are 8×8 so the HTML repr renders them as thumbnails; the spectra are 4-element vectors, small enough to print as values.

In [2]:
rng = np.random.default_rng(0)
n = 5
images = rng.random((n, 8, 8)).astype(np.float32)  # one 8x8 image per row
spectra = np.round(rng.random((n, 4)), 2)  # one 4-element vector per row

nf = NestedFrame(
    {
        "id": np.arange(n),
        "image": TensorExtensionArray.from_stack(images),
        "spectrum": TensorExtensionArray.from_stack(spectra),
        "flux": [[1.0, 2.0], [3.0], [4.0, 5.0, 6.0], [7.0], [8.0, 9.0]],
    },
    index=[10, 11, 12, 13, 14],
).nest_lists(["flux"], name="lc")
nf

id          image                  spectrum                         lc
10   0  [8×8] float32   [0.56, 0.48, 0.9, 0.09]  [{flux: 1.0}; …] (2 rows)
11   1  [8×8] float32   [0.7, 0.33, 0.18, 0.67]              [{flux: 3.0}]
12   2  [8×8] float32   [0.36, 0.33, 0.94, 0.2]  [{flux: 4.0}; …] (3 rows)
13   3  [8×8] float32  [0.51, 0.02, 0.16, 0.88]              [{flux: 7.0}]
14   4  [8×8] float32  [0.79, 0.56, 0.22, 0.56]  [{flux: 8.0}; …] (2 rows)

In [3]:
nf.dtypes

id                           int64
image        tensor[float, (8, 8)]
spectrum      tensor[double, (4,)]
lc          nested<flux: [double]>
dtype: object

Selecting a tensor column gives a `TensorSeries`. The dtype carries the shape and element type; `tensor_shape` is a shortcut to `dtype.shape` (`Series.shape` keeps its pandas meaning, the length of the series).

In [4]:
spectrum = nf["spectrum"]
type(spectrum).__name__, spectrum.dtype, spectrum.tensor_shape, spectrum.shape

('TensorSeries', tensor[double, (4,)], (4,), (5,))

In [5]:
spectrum

,spectrum
10,[0.56 0.48 0.9 0.09]
11,[0.7 0.33 0.18 0.67]
12,[0.36 0.33 0.94 0.2 ]
13,[0.51 0.02 0.16 0.88]
14,[0.79 0.56 0.22 0.56]


The HTML repr is the same one `NestedFrame` uses for a tensor column: values for small tensors, thumbnails for 2-d ones.

In [6]:
nf["image"]

,image
10,0.9870.0114
11,0.9860.0155
12,0.9830.0308
13,0.9910.042
14,0.9870.0254


A `TensorSeries` can also be built directly from a stack with `from_stack`, with the index and name, or from any `pd.Series` of tensor dtype.

In [7]:
TensorSeries.from_stack(spectra, index=nf.index, name="spectrum")

,spectrum
10,[0.56 0.48 0.9 0.09]
11,[0.7 0.33 0.18 0.67]
12,[0.36 0.33 0.94 0.2 ]
13,[0.51 0.02 0.16 0.88]
14,[0.79 0.56 0.22 0.56]


## Getting individual tensors and getting full array in other formats

Scalar access returns the tensor as a read-only numpy view over the arrow buffers, so nothing is copied.

In [8]:
first = spectrum[10]
first, type(first).__name__, first.flags.writeable

(array([0.56, 0.48, 0.9 , 0.09]), 'ndarray', False)

In [9]:
spectrum.iloc[-1]

array([0.79, 0.56, 0.22, 0.56])

`to_stack` returns the whole column as one `(n, *shape)` array. Without missing values it is a zero-copy view; here it even shares memory with the `spectra` array the column was built from.

In [10]:
stack = spectrum.to_stack()
stack.shape, stack.dtype, np.shares_memory(stack, spectra)

((5, 4), dtype('float64'), True)

Other formats: a Series of arrow fixed-size lists (`to_lists`, flattening each tensor, zero copy), an object array of tensors (`to_numpy`), and the pyarrow array itself.

In [11]:
spectrum.to_lists()

10    [0.56 0.48 0.9  0.09]
11    [0.7  0.33 0.18 0.67]
12    [0.36 0.33 0.94 0.2 ]
13    [0.51 0.02 0.16 0.88]
14    [0.79 0.56 0.22 0.56]
Name: spectrum, dtype: fixed_size_list<item: double>[4][pyarrow]

In [12]:
spectrum.to_numpy()[:2]

array([array([0.56, 0.48, 0.9 , 0.09]), array([0.7 , 0.33, 0.18, 0.67])],
      dtype=object)

In [13]:
spectrum.array.pa_array.type

FixedShapeTensorType(extension<arrow.fixed_shape_tensor[value_type=double, shape=[4]]>)

Missing rows are `pd.NA` on scalar access and are filled with `na_value` by `to_stack`.

In [14]:
with_missing = TensorSeries(TensorExtensionArray.from_sequence([spectra[0], None, spectra[2]]), name="spectrum")
with_missing[1], with_missing.to_stack(na_value=-1.0)

(<NA>,
 array([[ 0.56,  0.48,  0.9 ,  0.09],
        [-1.  , -1.  , -1.  , -1.  ],
        [ 0.36,  0.33,  0.94,  0.2 ]]))

----- not yet in PR -----

## Setting tensors in a series

pandas inspects a numpy array value before the extension array is consulted: a 2-d tensor looks like an `(N, 1)` column and trips an assertion, and a 1-d one looks like a sequence of values. `TensorSeries` rewraps a single tensor (any array with as many dimensions as the tensors) as a pyarrow tensor scalar, which pandas passes through untouched. So every assignment path works, and the tensor broadcasts to all the selected rows.

In [15]:
s = nf["spectrum"].copy()
one = np.full(4, -1.0)

s[10] = one  # by label
s.loc[11] = one * 2  # by label
s.iloc[2] = one * 3  # by position
s

,spectrum
10,[-1. -1. -1. -1.]
11,[-2. -2. -2. -2.]
12,[-3. -3. -3. -3.]
13,[0.51 0.02 0.16 0.88]
14,[0.79 0.56 0.22 0.56]


In [16]:
s[[13, 14]] = one * 4  # a list of labels: the tensor goes to every one of them
s[s.index <= 11] = one * 5  # a boolean mask
s.loc[11:12] = one * 6  # a label slice (inclusive)
s.iloc[3:] = one * 7  # a positional slice
s

,spectrum
10,[-5. -5. -5. -5.]
11,[-6. -6. -6. -6.]
12,[-6. -6. -6. -6.]
13,[-7. -7. -7. -7.]
14,[-7. -7. -7. -7.]


A stack or list of tensors, with one more dimension, still assigns one tensor per selected row.

In [17]:
s.iloc[:2] = np.zeros((2, 4))
s[[12, 13]] = [one, -one]
s

,spectrum
10,[0. 0. 0. 0.]
11,[0. 0. 0. 0.]
12,[-1. -1. -1. -1.]
13,[1. 1. 1. 1.]
14,[-7. -7. -7. -7.]


A tensor of the wrong shape is rejected before pandas sees it. `None` or `pd.NA` makes a row missing, and `fillna` takes a single tensor for every missing row.

In [18]:
try:
    s[10] = np.zeros(5)
except ValueError as e:
    print(type(e).__name__, e)

ValueError Expected a tensor of shape (4,), got shape (5,)


In [19]:
s[10] = None
s.loc[14] = pd.NA
s.fillna(np.full(4, 9.0))

,spectrum
10,[9. 9. 9. 9.]
11,[0. 0. 0. 0.]
12,[-1. -1. -1. -1.]
13,[1. 1. 1. 1.]
14,[9. 9. 9. 9.]


One thing that does not change: assigning to a label that is not in the index appends a row, and pandas builds that row as a plain object, so the result is an object-dtype Series. That is how pandas behaves for any extension dtype; use `pd.concat` with a `TensorSeries` to grow a tensor column.

In [20]:
grown = nf["spectrum"].copy()
grown.loc[99] = one
grown.dtype

dtype('O')

## Tensor math operations

Arithmetic operators apply tensor by tensor, computed with numpy on the `(n, *shape)` block, and the result is again a `TensorSeries`. The other operand may be a scalar, another tensor series, a single tensor, or one scalar per row.

In [21]:
spectrum * 2

,spectrum
10,[1.12 0.96 1.8 0.18]
11,[1.4 0.66 0.36 1.34]
12,[0.72 0.66 1.88 0.4 ]
13,[1.02 0.04 0.32 1.76]
14,[1.58 1.12 0.44 1.12]


In [22]:
(spectrum * 2).dtype, type(spectrum * 2).__name__

(tensor[double, (4,)], 'TensorSeries')

A common use is masking images. A single mask of the image shape, boolean or 0/1, multiplies every row. Numpy casts the booleans, so `True` keeps a pixel and `False` zeroes it. Here a circular aperture keeps the centre of each image.

In [23]:
yy, xx = np.mgrid[:8, :8]
aperture = (yy - 3.5) ** 2 + (xx - 3.5) ** 2 <= 3.5**2  # a boolean (8, 8) mask
aperture.astype(int)

array([[0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 1, 1, 1, 1, 0, 0],
       [0, 1, 1, 1, 1, 1, 1, 0],
       [0, 1, 1, 1, 1, 1, 1, 0],
       [0, 1, 1, 1, 1, 1, 1, 0],
       [0, 1, 1, 1, 1, 1, 1, 0],
       [0, 0, 1, 1, 1, 1, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0]])

In [24]:
masked = nf["image"] * aperture
masked

,image
10,0.9870
11,0.9860
12,0.9830
13,0.9140
14,0.9870


In [25]:
masked.to_stack()[0].round(2)

array([[0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.  , 0.82, 0.  , 0.86, 0.03, 0.  , 0.  ],
       [0.  , 0.54, 0.3 , 0.42, 0.03, 0.12, 0.67, 0.  ],
       [0.  , 0.38, 1.  , 0.98, 0.69, 0.65, 0.69, 0.  ],
       [0.  , 0.72, 0.53, 0.31, 0.49, 0.89, 0.93, 0.  ],
       [0.  , 0.32, 0.59, 0.34, 0.39, 0.89, 0.23, 0.  ],
       [0.  , 0.  , 0.79, 0.24, 0.88, 0.06, 0.  , 0.  ],
       [0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ]], dtype=float32)

The aperture flux per row is then a reduction over the tensor axes of the stack, giving an ordinary Series.

In [26]:
pd.Series(masked.to_stack().sum(axis=(1, 2)), index=nf.index, name="aperture_flux")

10    16.773943
11    20.461815
12    18.330360
13    16.151388
14    18.360607
Name: aperture_flux, dtype: float32

A series of masks, one per row, multiplies row by row. Boolean tensors are a `TensorSeries` like any other, built here from a stack or, later in this notebook, from an elementwise comparison.

In [27]:
per_row_masks = TensorSeries.from_stack(rng.random((n, 8, 8)) > 0.5, index=nf.index, name="mask")
per_row_masks.dtype

tensor[bool, (8, 8)]

In [28]:
nf["image"] * per_row_masks

,0
10,0.9580
11,0.9860
12,0.9720
13,0.9910
14,0.9130


Two tensor series are aligned on the index first, like any pandas arithmetic. Here the right operand has its index reversed, so each row still meets its own counterpart; rows present on only one side become missing.

In [29]:
reversed_index = spectrum.sort_index(ascending=False)
spectrum - reversed_index

,spectrum
10,[0. 0. 0. 0.]
11,[0. 0. 0. 0.]
12,[0. 0. 0. 0.]
13,[0. 0. 0. 0.]
14,[0. 0. 0. 0.]


In [30]:
spectrum + spectrum.iloc[:3]

,spectrum
10,[1.12 0.96 1.8 0.18]
11,[1.4 0.66 0.36 1.34]
12,[0.72 0.66 1.88 0.4 ]
13,<NA>
14,<NA>


A numpy array of the tensor shape applies to every row. A plain Series, or a 1-d array of one value per row, scales each row by its own value.

In [31]:
spectrum * np.array([1, 10, 100, 1000])

,spectrum
10,[ 0.56 4.8 90. 90. ]
11,[ 0.7 3.3 18. 670. ]
12,[ 0.36 3.3 94. 200. ]
13,[5.1e-01 2.0e-01 1.6e+01 8.8e+02]
14,[ 0.79 5.6 22. 560. ]


In [32]:
per_row = pd.Series([0, 1, 2, 3, 4], index=nf.index)
spectrum * per_row

,0
10,[0. 0. 0. 0.]
11,[0.7 0.33 0.18 0.67]
12,[0.72 0.66 1.88 0.4 ]
13,[1.53 0.06 0.48 2.64]
14,[3.16 2.24 0.88 2.24]


Tensors of different rank broadcast on their tensor shapes, numpy style: a series of 8-element rows scales each column of the 8×8 images.

In [33]:
column_weights = TensorSeries.from_stack(np.tile(np.linspace(0, 1, 8), (n, 1)), index=nf.index)
(nf["image"] * column_weights).tensor_shape

(8, 8)

The result dtype follows numpy, so integer tensors divide to doubles. Missing rows propagate, and a missing scalar operand makes every row missing. Unary operators, `divmod` and elementwise numpy ufuncs work too.

In [34]:
ints = TensorSeries.from_stack(np.arange(8).reshape(4, 2))
(ints * 2).dtype, (ints / 2).dtype

(tensor[int64, (2,)], tensor[double, (2,)])

In [35]:
with_missing + 1

,spectrum
0,[1.56 1.48 1.9 1.09]
1,<NA>
2,[1.36 1.33 1.94 1.2 ]


In [36]:
np.sqrt(spectrum)

,spectrum
10,[0.74833148 0.69282032 0.9486833 0.3 ]
11,[0.83666003 0.57445626 0.42426407 0.81853528]
12,[0.6 0.57445626 0.96953597 0.4472136 ]
13,[0.71414284 0.14142136 0.4 0.93808315]
14,[0.88881944 0.74833148 0.46904158 0.74833148]


In [37]:
-spectrum

,spectrum
10,[-0.56 -0.48 -0.9 -0.09]
11,[-0.7 -0.33 -0.18 -0.67]
12,[-0.36 -0.33 -0.94 -0.2 ]
13,[-0.51 -0.02 -0.16 -0.88]
14,[-0.79 -0.56 -0.22 -0.56]


In [38]:
quotient, remainder = divmod(spectrum, 0.25)
quotient

,spectrum
10,[2. 1. 3. 0.]
11,[2. 1. 0. 2.]
12,[1. 1. 3. 0.]
13,[2. 0. 0. 3.]
14,[3. 2. 0. 2.]


## Tensor boolean operations

pandas has firm expectations of what comparison operators return on an extension array, because it uses them itself:

- `==` and `!=` must return **one boolean per row**, a 1-d array of the series length. `in`, `Series.eq`, `isin`, `where`/`mask` with a condition, and the `assert_*_equal` testing utilities all build on it. A tensor-shaped result would break every one of those.
- Ordering operators (`<`, `<=`, `>`, `>=`) must raise `TypeError` for types without an ordering, so that sorting and the like fail cleanly rather than silently.

So `==` and `!=` compare **whole tensors**, row by row: equal if every element is equal, with a missing tensor unequal to everything (and `NaN` unequal to itself, as in numpy). The other operand can be a single tensor or another tensor series, aligned on the index.

In [39]:
spectrum == spectrum

10    True
11    True
12    True
13    True
14    True
Name: spectrum, dtype: bool

In [40]:
spectrum == spectra[1]  # a single tensor, compared to every row

10    False
11     True
12    False
13    False
14    False
Name: spectrum, dtype: bool

In [41]:
changed = spectrum.copy()
changed.iloc[2] = np.zeros(4)
spectrum != changed

10    False
11    False
12     True
13    False
14    False
Name: spectrum, dtype: bool

The result is an ordinary boolean Series, so it works wherever pandas takes a row mask: selecting the rows of the series, or of the whole frame, and `where`/`mask`, which replace the other rows with a missing value or with a tensor.

In [42]:
changed_rows = spectrum != changed
spectrum[changed_rows]

,spectrum
12,[0.36 0.33 0.94 0.2 ]


In [43]:
nf[changed_rows]

id          image                 spectrum                         lc
12   2  [8×8] float32  [0.36, 0.33, 0.94, 0.2]  [{flux: 4.0}; …] (3 rows)

In [44]:
spectrum.where(~changed_rows)  # keep the unchanged rows, the rest missing

,spectrum
10,[0.56 0.48 0.9 0.09]
11,[0.7 0.33 0.18 0.67]
12,<NA>
13,[0.51 0.02 0.16 0.88]
14,[0.79 0.56 0.22 0.56]


In [45]:
spectrum.mask(changed_rows, np.zeros(4))  # replace the changed rows with a tensor

,spectrum
10,[0.56 0.48 0.9 0.09]
11,[0.7 0.33 0.18 0.67]
12,[0. 0. 0. 0.]
13,[0.51 0.02 0.16 0.88]
14,[0.79 0.56 0.22 0.56]


In [46]:
spectra[3] in spectrum.array, np.zeros(4) in spectrum.array

(True, False)

In [47]:
try:
    spectrum < changed
except TypeError as e:
    print(type(e).__name__, e)

TypeError 'lt' is not supported between tensors, which have no ordering. For an elementwise comparison giving boolean tensors use the numpy ufunc, e.g. np.less(a, b).


For **elementwise** comparisons, giving boolean tensors, use the numpy comparison ufuncs: `np.equal`, `np.not_equal`, `np.less`, `np.less_equal`, `np.greater`, `np.greater_equal`. (On a plain Series pandas would reroute these to the row-wise `==`; `TensorSeries` applies them to the arrays instead, so two series must share an index.)

In [48]:
bright = np.greater(spectrum, 0.5)
bright

,spectrum
10,[ True False True False]
11,[ True False False True]
12,[False False True False]
13,[ True False False True]
14,[ True True False True]


In [49]:
bright.dtype

tensor[bool, (4,)]

Boolean tensors combine with `&`, `|` and `~`, and multiply into masks for math.

In [50]:
band = np.greater(spectrum, 0.3) & ~np.greater(spectrum, 0.7)
band

,spectrum
10,[ True True False False]
11,[ True True False True]
12,[ True True False False]
13,[ True False False False]
14,[False True False True]


In [51]:
spectrum * band  # keep the values in the band, zero the rest

,spectrum
10,[0.56 0.48 0. 0. ]
11,[0.7 0.33 0. 0.67]
12,[0.36 0.33 0. 0. ]
13,[0.51 0. 0. 0. ]
14,[0. 0.56 0. 0.56]


To go from an elementwise comparison back to one boolean per row, reduce the numpy stack over the tensor axes. That gives an ordinary boolean Series, which filters the frame.

In [52]:
any_bright = pd.Series(np.greater(nf["image"], 0.98).to_stack().any(axis=(1, 2)), index=nf.index)
nf[any_bright]

id          image                  spectrum                         lc
10   0  [8×8] float32   [0.56, 0.48, 0.9, 0.09]  [{flux: 1.0}; …] (2 rows)
11   1  [8×8] float32   [0.7, 0.33, 0.18, 0.67]              [{flux: 3.0}]
12   2  [8×8] float32   [0.36, 0.33, 0.94, 0.2]  [{flux: 4.0}; …] (3 rows)
13   3  [8×8] float32  [0.51, 0.02, 0.16, 0.88]              [{flux: 7.0}]
14   4  [8×8] float32  [0.79, 0.56, 0.22, 0.56]  [{flux: 8.0}; …] (2 rows)

Limitations, all following from what pandas does before the extension array is reached:

- `spectrum.eq(tensor)` and the other flex methods length-check a numpy array operand and raise; use the operator form `spectrum == tensor`.
- With the numpy array on the left, `tensor == spectrum` still goes through pandas and is row-wise, but `tensor == spectrum.array` at the array level is handled by numpy and is elementwise.
- Anything that hashes elements (`unique`, `value_counts`, `isin`, `duplicated`, merging on a tensor column) does not work, since numpy arrays are not hashable. Same as for nested columns.

## Series slicing

Should this be added to nested too?

A `TensorSeries` defines pandas' `_constructor` hook to dispatch on the result dtype: any operation whose result is still a tensor column (slicing, filtering, `take`, `copy`, sorting, `concat`) returns a `TensorSeries`, while a result of another dtype (`isna`, `map`) is a plain `Series`, so you never hold a `TensorSeries` whose methods cannot work.

In [53]:
results = {
    "iloc[1:]": spectrum.iloc[1:],
    "boolean mask": spectrum[spectrum.index > 11],
    "head(2)": spectrum.head(2),
    "sort_index": spectrum.sort_index(ascending=False),
    "concat": pd.concat([spectrum, spectrum]),
    "isna()": spectrum.isna(),
    "map(np.sum)": spectrum.map(np.sum),
}
pd.Series({name: type(value).__name__ for name, value in results.items()})

iloc[1:]        TensorSeries
boolean mask    TensorSeries
head(2)         TensorSeries
sort_index      TensorSeries
concat          TensorSeries
isna()                Series
map(np.sum)           Series
dtype: str

`NestedSeries` does not define the hook, so slicing one gives a plain `Series`, which loses the `NestedSeries` methods such as `explode` and `to_lists`. The `.nest` accessor is registered on every Series of nested dtype, so that still works.

In [54]:
lc = nf["lc"]
type(lc).__name__, type(lc.iloc[1:]).__name__

('NestedSeries', 'Series')

In [55]:
try:
    lc.iloc[1:].explode()
except AttributeError as e:
    print(type(e).__name__, e)

In [56]:
lc.iloc[1:].nest.to_flat()

,flux
11,3.0
12,4.0
12,5.0
12,6.0
13,7.0
14,8.0
14,9.0


## Alternative DataFrame Series Selection

`NestedFrame.__getitem__` wraps a tensor column in a `TensorSeries` (and a nested column in a `NestedSeries`). Attribute access, `.loc` with a column label and `pop` all route through it.

In [57]:
pd.Series(
    {
        'nf["image"]': type(nf["image"]).__name__,
        "nf.image": type(nf.image).__name__,
        'nf.loc[:, "image"]': type(nf.loc[:, "image"]).__name__,
        'nf.pop("image")': type(nf.copy().pop("image")).__name__,
        'nf["lc"]': type(nf["lc"]).__name__,
    }
)

nf["image"]           TensorSeries
nf.image              TensorSeries
nf.loc[:, "image"]    TensorSeries
nf.pop("image")       TensorSeries
nf["lc"]              NestedSeries
dtype: str

Other paths build the column through pandas' internal `_constructor_sliced` hook, which `NestedFrame` does not override, so they return a plain `Series` for both tensor and nested columns: positional column access with `iloc`, iterating with `items()` (which also drives `apply` over columns and `iterrows`), and `squeeze`.

In [58]:
col = nf.columns.get_loc("image")
pd.Series(
    {
        "nf.iloc[:, col]": type(nf.iloc[:, col]).__name__,
        'dict(nf.items())["image"]': type(dict(nf.items())["image"]).__name__,
        'nf[["image"]].squeeze(axis=1)': type(nf[["image"]].squeeze(axis=1)).__name__,
        'nf.iloc[:, nf.columns.get_loc("lc")]': type(nf.iloc[:, nf.columns.get_loc("lc")]).__name__,
    }
)

nf.iloc[:, col]                         Series
dict(nf.items())["image"]               Series
nf[["image"]].squeeze(axis=1)           Series
nf.iloc[:, nf.columns.get_loc("lc")]    Series
dtype: str

The data is the same, but the `TensorSeries` and `NestedSeries` methods are missing on those results.

In [59]:
try:
    nf.iloc[:, col].to_stack()
except AttributeError as e:
    print(type(e).__name__, e)

AttributeError 'Series' object has no attribute 'to_stack'


Until the hook is wired up (issue #541), wrap the result yourself. The dtype is kept, so this is free.

In [60]:
TensorSeries(nf.iloc[:, col]).to_stack().shape

(5, 8, 8)